# NER Learning Model Test
Entities: `topic`, `subtopic`, `duration`, `start_date`, `person`, `date`

In [ ]:
import torch
from transformers import DistilBertTokenizerFast, DistilBertForTokenClassification

MODEL_PATH = r'D:\FYP_20March\Eva_AI_Server\models\ner2_model'

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_PATH)
model     = DistilBertForTokenClassification.from_pretrained(MODEL_PATH)
model.eval()

id2label = {int(k): v for k, v in model.config.id2label.items()}
print('Model loaded.')
print('Labels:', list(id2label.values()))

In [ ]:
def extract_entities(text, model, tokenizer, id2label):
    words = text.lower().split()
    enc   = tokenizer(
        words,
        is_split_into_words = True,
        return_tensors      = 'pt',
        truncation          = True,
        max_length          = 128,
    )
    with torch.no_grad():
        logits = model(**enc).logits
    pred_ids = logits.argmax(dim=-1).squeeze(0).tolist()
    word_ids = enc.word_ids(batch_index=0)

    word_labels = {}
    for tok_idx, word_id in enumerate(word_ids):
        if word_id is None or word_id in word_labels:
            continue
        word_labels[word_id] = id2label.get(pred_ids[tok_idx], 'O')

    entities = []
    current_entity = None
    current_words  = []
    for wi in range(len(words)):
        label = word_labels.get(wi, 'O')
        if label.startswith('B-'):
            if current_entity:
                entities.append((' '.join(current_words), current_entity))
            current_entity = label[2:]
            current_words  = [words[wi]]
        elif label.startswith('I-') and current_entity == label[2:]:
            current_words.append(words[wi])
        else:
            if current_entity:
                entities.append((' '.join(current_words), current_entity))
            current_entity = None
            current_words  = []
    if current_entity:
        entities.append((' '.join(current_words), current_entity))

    return entities

print('extract_entities() ready.')

In [ ]:
# ── Add your test inputs here ──────────────────────────────────────
inputs = [
    "create a python roadmap for 2 weeks",
    "learn django oop starting next monday",
    "i want to study javascript for 1 month",
    "start learning react from tomorrow",
    "delete my python learning plan",
    "remove javascript roadmap",
    "pause my python learning",
    "reschedule my python session to friday",
    "learn python with Ali for 3 weeks",
    "create java roadmap for 2 weeks starting monday",
    "2 weeks python starting next monday",
    "python 2 weeks",
]

In [ ]:
for text in inputs:
    entities = extract_entities(text, model, tokenizer, id2label)
    print(f'Input : "{text}"')
    if entities:
        for value, entity in entities:
            print(f'  {entity:<20}: "{value}"')
    else:
        print('  (no entities found)')
    print()

In [ ]:
# ── Position swap test ─────────────────────────────────────────────
swap_groups = [
    [
        "create a python roadmap for 2 weeks",
        "python 2 weeks roadmap",
        "2 weeks python",
    ],
    [
        "learn python oop starting next monday",
        "next monday start python oop",
        "python oop next monday",
    ],
]

for group in swap_groups:
    print('=' * 60)
    for text in group:
        entities = extract_entities(text, model, tokenizer, id2label)
        ent_str  = ', '.join(f'{e}="{v}"' for v, e in entities) if entities else '(none)'
        print(f'  "{text}"')
        print(f'  -> {ent_str}')
    print()